<a href="https://colab.research.google.com/github/phamtuanlinh227-collab/python_for_chemistry/blob/master/Weekend-Projects%20Phase%202/03_Training_2_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121

!pip install dgl -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html


!pip install torchdata==0.7.1 dgllife
!pip install --pre deepchem

Looking in indexes: https://download.pytorch.org/whl/cu121
Looking in links: https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html


In [4]:
!pip install torch_geometric
!pip install deepchem
!pip install rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import AllChem
from deepchem.models.torch_models import DMPNNModel
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# === TRAIN CON AI DỰ ĐOÁN BBBP TRƯỚC BẰNG RANDOMFOREST ===
# CÀO DATA TỪ FILE BBBP
print("🌲 System startup process")
tasks, datasets, transformers = dc.molnet.load_bbbp(featurizer="ECFP", splitter="scaffold")
train_dataset, valid_dataset, test_dataset = datasets

# GỘP TỆP TRAIN VỚI VALID LẠI
print(f"TRAIN DATASET {len(train_dataset)}  | VALID DATASET {len(valid_dataset)} COMPOUNDS")
print("Merge two files ⚡")

# merged 2 file for AI RandomForest training with a 9:1 ratio

X_train_merged = np.concatenate([train_dataset.X, valid_dataset.X])
y_train_merged = np.concatenate([train_dataset.y, valid_dataset.y])
# Ép phẳng
y_train_merged = y_train_merged.flatten()
print(f"📝 Merged success {len(X_train_merged)} compounds ")
# Train con AI randomforest thôi
model_bbbp = RandomForestClassifier(n_estimators=100, random_state=42)
model_bbbp.fit(X_train_merged, y_train_merged)

print("Use AI Model for test dataset")
X_test = test_dataset.X
y_test = test_dataset.y.flatten()

y_prob = model_bbbp.predict_proba(X_test)[:,1]

score_bbbp = roc_auc_score(y_test, y_prob)
print("="*40)
print(f"ROC-AUC-SCORE: {score_bbbp:.3f}")
print("="*40)


# === TRAIN CON AI TỪ FILE TOX21 ===
print("🚬Load file tox21 for AI training")
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=dc.feat.DMPNNFeaturizer(), splitter="scaffold")
train_dataset_1, valid_dataset_1, test_dataset_1 = datasets

print("Dowload successfully")
print(f"Train dataset: {len(train_dataset)} compounds")

model_tox21 = dc.models.torch_models.DMPNNModel(
    n_tasks = 12,
    mode = 'classification',
    dropout = 0.2,
    learning_rate = 0.003,
    hidden_size = 64,
    depth = 3,
    batch_size = 128
)
print("Done! Let's traning AI")
class HardcoreRadarCallback:
  def __init__(self, valid_dataset_1, interval):
    self.valid_dataset_1 = valid_dataset_1
    self.interval = interval
    self.best_score = 0.0

  def __call__(self, model, step, **kwargs):
    if step % self.interval == 0:
      print(f"[STEP {step}] the system is processing data")
      y_pred = model_tox21.predict(self.valid_dataset_1)
      y_true = np.squeeze(self.valid_dataset_1.y)

      # Extract probability for the positive class depending on prediction shape
      if len(y_pred.shape) == 3:
        y_pred_probs = y_pred[:, :, 1]
      elif len(y_pred.shape) == 2:
        y_pred_probs = y_pred[:, 1]
      else:
        y_pred_probs = y_pred
      score_tox21 = roc_auc_score(y_true, y_pred_probs)
      print(f"Score: {score_tox21:.4f}")
      if score_tox21 > self.best_score:
        self.best_score = score_tox21
        model_tox21.save_checkpoint()
        print("🏆Wow the highest score, this was save")
radar = HardcoreRadarCallback(valid_dataset_1, interval=49)
model_tox21.fit(train_dataset_1, nb_epoch=30, callbacks=[radar])

print("Shut down, AI traning is successful")

🌲 System startup process
TRAIN DATASET 1631  | VALID DATASET 204 COMPOUNDS
Merge two files ⚡
📝 Merged success 1835 compounds 
Use AI Model for test dataset
ROC-AUC-SCORE: 0.740
🚬Load file tox21 for AI training
Dowload successfully
Train dataset: 1631 compounds
Done! Let's traning AI
[STEP 49] the system is processing data
Score: 0.6433
🏆Wow the highest score, this was save
[STEP 98] the system is processing data
Score: 0.6598
🏆Wow the highest score, this was save
[STEP 147] the system is processing data
Score: 0.7022
🏆Wow the highest score, this was save
[STEP 196] the system is processing data
Score: 0.7126
🏆Wow the highest score, this was save
[STEP 245] the system is processing data
Score: 0.7237
🏆Wow the highest score, this was save
[STEP 294] the system is processing data
Score: 0.7276
🏆Wow the highest score, this was save
[STEP 343] the system is processing data
Score: 0.7262
[STEP 392] the system is processing data
Score: 0.7320
🏆Wow the highest score, this was save
[STEP 441] t